# M7 Validation: Real RGB Perception + LLM Plan + LIBERO-10 Task 0 (Linux/Colab)

## 1 — Secrets & clone

In [1]:
import getpass, os

github_token = getpass.getpass('GitHub PAT (repo scope): ').strip()
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ').strip()
assert github_token != '', f'github_token was not set'

In [2]:
import os, subprocess, sys

REPO_DIR = '/content/Longor'
GITHUB_REPO = os.environ.get('GITHUB_REPO', 'pmish11/Longor-Robotics')

if os.path.exists(f'{REPO_DIR}/.git'):
    print('Existing clone detected -> git pull')
    subprocess.run(['git', 'pull', '--ff-only'], check=True, cwd=REPO_DIR)
else:
    clone_url = f"https://{github_token}@github.com/{GITHUB_REPO}.git"
    subprocess.run(['git', 'clone', clone_url, REPO_DIR], check=True)
    print('Cloned OK')

os.chdir(REPO_DIR)

Cloned OK


## 2 — App deps + LIBERO + headless

In [3]:
import importlib.util
import logging
import os
import subprocess
import sys

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

# Colab images can have stale apt indexes; refresh first to avoid mesa package 404s.
!apt-get update -y
!apt-get install -y --fix-missing cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r deps/requirements-libero.txt

spec = importlib.util.find_spec('robosuite')
if spec:
    script = spec.submodule_search_locations[0] + '/scripts/setup_macros.py'
    result = subprocess.run([sys.executable, script], check=False)
    if result.returncode != 0:
        logging.warning('robosuite setup_macros.py failed (rc=%s); continuing', result.returncode)

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,840 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]          
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.5 MB]  
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1

## 3 — SmolVLA VLA venv

In [4]:
import os, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    result = subprocess.run(
        [VLA_PYTHON, '-m', 'pip', *args],
        capture_output=True, text=True,
    )
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, result.args)

%pip install --quiet virtualenv
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
else:
    print('.vla-venv already exists, skipping creation.')
vla_pip('install', '-U', 'pip')
vla_pip('install', '-r', '/content/Longor/deps/requirements-vla-smolvla-linux.txt')
print('SmolVLA VLA venv ready.')


ze=13782 sha256=804cd15fd6f72fd2d0fdd43bcdc392b2d9100ff21eea9ac5093fd3e19be83cc8
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


SmolVLA VLA venv ready.


## 4 — Env config (M7: PERCEPTION_BACKEND=real)

In [5]:
import importlib.util, os, yaml

env_content = f"""OPENAI_API_KEY={os.environ['OPENAI_API_KEY']}
PERCEPTION_BACKEND=real
LLM_MODEL=gpt-4.1-mini
LOG_LEVEL=DEBUG
VLA_CHUNK_STEPS=10
VLA_STEPS_PER_ACTION=1
VLA_ACTION_SCALE=1.0
LIBERO_EE_DELTA_SCALE=1.0
"""
with open('/content/Longor/.env', 'w') as f:
    f.write(env_content)
print('Wrote /content/Longor/.env (PERCEPTION_BACKEND=real, VLA_CHUNK_STEPS=10)')

libero_config_path = os.environ.get('LIBERO_CONFIG_PATH', os.path.expanduser('~/.libero'))
config_file = os.path.join(libero_config_path, 'config.yaml')

if not os.path.exists(config_file):
    os.makedirs(libero_config_path, exist_ok=True)
    spec = importlib.util.find_spec('libero')
    libero_pkg = os.path.join(os.path.dirname(spec.origin), 'libero')
    default_config = {
        'benchmark_root': libero_pkg,
        'bddl_files': os.path.join(libero_pkg, 'bddl_files'),
        'init_states': os.path.join(libero_pkg, 'init_files'),
        'datasets': os.path.join(os.path.dirname(libero_pkg), 'datasets'),
        'assets': os.path.join(libero_pkg, 'assets'),
    }
    with open(config_file, 'w') as f:
        yaml.dump(default_config, f)
    print(f'Created {config_file}')
else:
    print(f'Already exists: {config_file}')

Wrote /content/Longor/.env (PERCEPTION_BACKEND=real, VLA_CHUNK_STEPS=10)
Created /root/.libero/config.yaml


## 5 — Start SmolVLA VLA server + health

In [6]:
import json, os, subprocess, time, urllib.request

VLA_LOG = '/tmp/m7_vla_smolvla_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'
POLL_SEC = 15
MAX_WAIT_SEC = int(os.environ.get('VLA_STARTUP_MAX_WAIT_SEC', 120))
MAX_STALL_SEC = int(os.environ.get('VLA_STARTUP_MAX_STALL_SEC', 60))
VLA_STALE_PATTERNS = ('run_vla_server.py', 'smolvla_service.py')


def _read_log_snippet(n=40):
    try:
        with open(VLA_LOG) as f:
            lines = f.readlines()
        return ''.join(lines[-n:])
    except FileNotFoundError:
        return '(log file not created yet)'


def _stop_vla(proc, log_handle):
    if log_handle and not log_handle.closed:
        log_handle.flush()
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)
    if log_handle and not log_handle.closed:
        log_handle.close()


def _startup_failure(reason, proc, log_handle, start_time):
    elapsed = time.monotonic() - start_time
    snippet = _read_log_snippet(40)
    _stop_vla(proc, log_handle)
    hints = (
        'Likely causes: HF_TOKEN missing/invalid (checkpoint download fails), '
        'very long first-time checkpoint download, or GPU OOM during model load. '
        f'Check HF_TOKEN, disk space, and full log at {VLA_LOG}.'
    )
    raise RuntimeError(
        f'{reason} after {elapsed:.0f}s.\n'
        f'Recent log:\n{snippet}\n'
        f'{hints}'
    )


def _print_new_log(log_handle, log_offset):
    try:
        log_handle.flush()
        with open(VLA_LOG) as f:
            f.seek(log_offset)
            new_chunk = f.read()
            new_offset = f.tell()
        if new_chunk:
            print(new_chunk, end='')
        return new_offset, bool(new_chunk)
    except FileNotFoundError:
        return log_offset, False


def _cleanup_stale_vla_processes():
    port_result = subprocess.run(
        ['fuser', '-k', '8765/tcp'], capture_output=True, text=True,
    )
    port_killed = bool(port_result.stdout.strip() or port_result.stderr.strip())

    stale_pids = []
    for pattern in VLA_STALE_PATTERNS:
        pgrep = subprocess.run(
            ['pgrep', '-f', pattern], capture_output=True, text=True,
        )
        for line in pgrep.stdout.splitlines():
            pid = line.strip()
            if pid.isdigit():
                stale_pids.append(int(pid))

    stale_pids = sorted(set(stale_pids))
    for pid in stale_pids:
        subprocess.run(['kill', '-9', str(pid)], capture_output=True)

    if port_killed or stale_pids:
        print(
            f'Cleanup: port8765={port_killed}, '
            f'stale_pids={stale_pids or "none"}'
        )
    else:
        print('Cleanup: no stale VLA processes found')


_cleanup_stale_vla_processes()
print(
    f'Startup limits: hard={MAX_WAIT_SEC}s, stall={MAX_STALL_SEC}s '
    f'(override via VLA_STARTUP_MAX_WAIT_SEC / VLA_STARTUP_MAX_STALL_SEC)'
)

log = open(VLA_LOG, 'w', buffering=1)
# Server-side action queue batch size (matches libero_smoke apply_smoke_env_defaults).
VLA_CHUNK_STEPS = '10'
env_for_server = {
    **os.environ,
    'PYTHONUNBUFFERED': '1',
    'VLA_POLICY': 'smolvla',
    'SMOLVLA_CHECKPOINT': 'HuggingFaceVLA/smolvla_libero',
    'VLA_CHUNK_STEPS': VLA_CHUNK_STEPS,
}
print(f'Server queue env: VLA_CHUNK_STEPS={VLA_CHUNK_STEPS} (actions per /predict)')
vla_proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    stdout=log, stderr=subprocess.STDOUT, cwd='/content/Longor',
    env=env_for_server,
)

start_time = time.monotonic()
log_offset = 0
last_log_progress = start_time
health_ok = False

try:
    while time.monotonic() - start_time < MAX_WAIT_SEC:
        time.sleep(POLL_SEC)
        elapsed = time.monotonic() - start_time

        if vla_proc.poll() is not None:
            log_offset, _ = _print_new_log(log, log_offset)
            _startup_failure(
                f'VLA server exited early (code {vla_proc.returncode})',
                vla_proc, log, start_time,
            )

        log_offset, had_new_log = _print_new_log(log, log_offset)
        if had_new_log:
            last_log_progress = time.monotonic()

        try:
            resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
            body = json.loads(resp.read())
            log_offset, _ = _print_new_log(log, log_offset)
            print(f'\nVLA server ready ({elapsed:.0f}s):', body)
            assert body.get('model') == 'smolvla', body
            health_ok = True
            break
        except Exception:
            print(f'Status: waiting for /health ({elapsed:.0f}s)')
            if time.monotonic() - last_log_progress >= MAX_STALL_SEC:
                log_offset, _ = _print_new_log(log, log_offset)
                _startup_failure(
                    f'[stall timeout] VLA server startup stalled '
                    f'(no new log output for {MAX_STALL_SEC}s; '
                    f'global cap {MAX_WAIT_SEC}s)',
                    vla_proc, log, start_time,
                )

    if not health_ok:
        log_offset, _ = _print_new_log(log, log_offset)
        _startup_failure(
            f'[hard timeout] VLA server did not become healthy within '
            f'{MAX_WAIT_SEC}s',
            vla_proc, log, start_time,
        )
except KeyboardInterrupt:
    print('Startup interrupted; stopping VLA server...')
    raise
finally:
    if not health_ok:
        _stop_vla(vla_proc, log)

Cleanup: no stale VLA processes found
Startup limits: hard=120s, stall=60s (override via VLA_STARTUP_MAX_WAIT_SEC / VLA_STARTUP_MAX_STALL_SEC)
Server queue env: VLA_CHUNK_STEPS=10 (actions per /predict)
INFO:     Started server process [3515]
INFO:     Waiting for application startup.
2026-07-18 15:49:48,168 INFO Loading SmolVLA checkpoint=HuggingFaceVLA/smolvla_libero ...
2026-07-18 15:49:56,283 INFO Loading SmolVLA checkpoint=HuggingFaceVLA/smolvla_libero on cuda ...
2026-07-18 15:49:56,611 INFO HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-07-18 15:49:56,845 INFO HTTP Request: HEAD https://huggingface.co/HuggingFaceVLA/smolvla_libero/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-18 15:49:56,853 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/HuggingFaceVLA/smolvla_libero/6721902bc4d61e50a3bfdb11dfb4cb626f05d102/config.json "HTTP/1.1 200 OK"
2026-07-18 15:49:56,862 INFO HTTP Request: GET https://huggin

## 6 — Gold vs Longor sweep (n_action_steps matrix)

Bounded experiment: official `lerobot-eval` vs Longor `test_grasp_smoke` on **LIBERO-10 task 0**.

**Defaults (T4-friendly):** sweep `n_action_steps` ∈ {10, 5} first, **5 episodes/runs per config**. Config `n=1` is skipped unless you set `GOLD_SWEEP_INCLUDE_N1=1` (very slow).

Longor uses parity env from `apply_smoke_env_defaults` (256×256 obs, image flip, SmolVLA state mode). Server `VLA_CHUNK_STEPS` restarts to match each gold `policy.n_action_steps`.

**VRAM:** Gold eval loads the policy in-process — stop §5 server before §6a; §6b restarts the server per Longor config.

**Resilience:** §6a streams each eval to a per-config log, catches timeouts, and writes `gold_json` after every config so a VM kill does not lose the whole matrix.


In [7]:
import json
import os
import re
import shutil
import subprocess
from dataclasses import dataclass
from datetime import datetime

# Fast configs first; n=1 is optional (inference every env step — hours on T4).
GOLD_SWEEP_STEPS = [int(x) for x in os.environ.get('GOLD_SWEEP_STEPS', '10,5').split(',') if x.strip()]
N_EPISODES = int(os.environ.get('GOLD_N_EPISODES', '5'))
if os.environ.get('GOLD_SWEEP_INCLUDE_N1', '').strip().lower() in ('1', 'true', 'yes'):
    if 1 not in GOLD_SWEEP_STEPS:
        GOLD_SWEEP_STEPS.append(1)

TIMEOUT_BY_N = {10: 1800, 5: 2700, 1: 7200}
EPISODES_BY_N = {1: min(N_EPISODES, 3)}

os.environ['M7_SWEEP_STEPS'] = ','.join(str(n) for n in GOLD_SWEEP_STEPS)
os.environ['M7_N_EPISODES'] = str(N_EPISODES)
os.environ['M7_N_LONGOR_RUNS'] = os.environ.get('M7_N_LONGOR_RUNS', str(N_EPISODES))

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
gold_log = f'{trace_dir}/gold_lerobot_eval_sweep_{stamp}.txt'
gold_json = f'{trace_dir}/gold_lerobot_eval_sweep_{stamp}.json'
os.environ['GOLD_EVAL_LOG'] = gold_log
os.environ['GOLD_EVAL_JSON'] = gold_json

vla_python = '/content/Longor/.vla-venv/bin/python'
vla_bin = '/content/Longor/.vla-venv/bin'

sub_env = {
    **os.environ,
    'MUJOCO_GL': 'egl',
    'PYOPENGL_PLATFORM': 'egl',
    'MPLBACKEND': 'Agg',
    'HF_HUB_DISABLE_PROGRESS_BARS': '0',
    'TQDM_DISABLE': '0',
    'PYTHONUNBUFFERED': '1',
}

try:
    _stop_vla(vla_proc, log)
except NameError:
    pass
try:
    _cleanup_stale_vla_processes()
except NameError:
    subprocess.run(['fuser', '-k', '8765/tcp'], capture_output=True)

logs: list[str] = []
gold_matrix: dict[str, dict] = {}


@dataclass
class EvalResult:
    returncode: int
    stdout: str
    timed_out: bool
    log_path: str


def _episodes_for(n_steps: int) -> int:
    return EPISODES_BY_N.get(n_steps, N_EPISODES)


def _timeout_for(n_steps: int) -> int:
    return TIMEOUT_BY_N.get(n_steps, 2700)


def _parse_gold_success(text: str, expected_episodes: int) -> tuple[int | None, int | None, float | None, str]:
    if not text:
        return None, None, None, 'empty_log'
    candidates: list[tuple[int, int, int, str]] = []
    for m in re.finditer(
        r'(?:success(?:_count|es)?|solved)\s*[=:]?\s*(\d+)\s*/\s*(\d+)',
        text,
        re.I,
    ):
        num, den = int(m.group(1)), int(m.group(2))
        if den > 0:
            candidates.append((m.start(), num, den, f'success_count={num}/{den}'))
    for m in re.finditer(r'success(?:_rate)?\s*[=:]?\s*([0-9.]+)\s*%?', text, re.I):
        value = float(m.group(1))
        rate = value / 100.0 if value > 1.0 else value
        den = expected_episodes
        num = int(round(rate * den))
        candidates.append((m.start(), num, den, f'success_rate={value}'))
    if not candidates:
        return None, None, None, 'success_metric_not_found'
    _, num, den, source = sorted(candidates, key=lambda x: x[0])[-1]
    rate = num / den if den else None
    return num, den, rate, source


def _parse_gold_from_output_dir(output_dir: str, expected_episodes: int) -> tuple[int | None, int | None, float | None, str]:
    import glob

    if not os.path.isdir(output_dir):
        return None, None, None, 'output_dir_missing'
    for path in sorted(glob.glob(f'{output_dir}/**/*.json', recursive=True)):
        try:
            with open(path, encoding='utf-8') as f:
                data = json.load(f)
        except (OSError, json.JSONDecodeError):
            continue
        stack = [data]
        while stack:
            node = stack.pop()
            if isinstance(node, dict):
                for key in ('success_rate', 'avg_sum_reward', 'pc_success'):
                    if key in node and isinstance(node[key], (int, float)):
                        rate = float(node[key])
                        if key != 'success_rate' and rate > 1.0:
                            rate = rate / 100.0
                        den = expected_episodes
                        num = int(round(rate * den))
                        return num, den, rate, f'{os.path.basename(path)}:{key}'
                stack.extend(node.values())
            elif isinstance(node, list):
                stack.extend(node)
    return None, None, None, 'output_dir_no_metric'


def _run_gold_eval(cmd: list[str], log_path: str, timeout_sec: int) -> EvalResult:
    os.makedirs(os.path.dirname(log_path) or '.', exist_ok=True)
    with open(log_path, 'w', encoding='utf-8', buffering=1) as out_f:
        proc = subprocess.Popen(
            cmd,
            cwd='/content/Longor',
            env=sub_env,
            stdout=out_f,
            stderr=subprocess.STDOUT,
            text=True,
        )
        timed_out = False
        try:
            proc.wait(timeout=timeout_sec)
        except subprocess.TimeoutExpired:
            timed_out = True
            proc.kill()
            try:
                proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                pass
            out_f.write(f'\n\n=== subprocess TIMEOUT after {timeout_sec}s ===\n')
    with open(log_path, encoding='utf-8') as f:
        combined = f.read()
    rc = -9 if timed_out else (proc.returncode if proc.returncode is not None else -1)
    return EvalResult(returncode=rc, stdout=combined, timed_out=timed_out, log_path=log_path)


def _print_log_tail(path: str, n: int = 12) -> None:
    if not os.path.exists(path):
        return
    with open(path, encoding='utf-8') as f:
        lines = f.readlines()
    print(f'--- tail {path} ({min(n, len(lines))} lines) ---')
    for line in lines[-n:]:
        print(line.rstrip())


def _write_gold_artifacts() -> None:
    with open(gold_log, 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(logs))
    with open(gold_json, 'w', encoding='utf-8') as f:
        json.dump(
            {
                'stamp': stamp,
                'n_episodes_default': N_EPISODES,
                'sweep_steps': GOLD_SWEEP_STEPS,
                'configs': gold_matrix,
            },
            f,
            indent=2,
        )


print(
    f'Gold sweep: steps={GOLD_SWEEP_STEPS} n_episodes={N_EPISODES} '
    f'(n=1 capped at {EPISODES_BY_N.get(1, N_EPISODES)} when enabled)'
)

deps_cmd = [vla_python, '-m', 'pip', 'install', '-q', 'lerobot[libero]>=0.5']
print(f'Deps: {" ".join(deps_cmd)}')
deps = subprocess.run(deps_cmd, cwd='/content/Longor', env=sub_env, capture_output=True, text=True, timeout=1200)
logs.append(
    f'=== deps install ===\ncommand: {" ".join(deps_cmd)}\nexit_code: {deps.returncode}\n\n'
    f'--- stdout ---\n{deps.stdout}\n--- stderr ---\n{deps.stderr}\n'
)
_write_gold_artifacts()
if deps.returncode != 0:
    raise RuntimeError(f'Failed to prepare .vla-venv eval deps (exit={deps.returncode})')

for n_steps in GOLD_SWEEP_STEPS:
    n_ep = _episodes_for(n_steps)
    timeout_sec = _timeout_for(n_steps)
    eval_out = f'{trace_dir}/gold_lerobot_eval_n{n_steps}_{stamp}'
    per_log = f'{trace_dir}/gold_lerobot_eval_n{n_steps}_{stamp}.log'
    base_args = [
        '--policy.path=HuggingFaceVLA/smolvla_libero',
        f'--policy.n_action_steps={n_steps}',
        '--policy.device=cuda',
        '--env.type=libero',
        '--env.task=libero_10',
        '--env.task_ids=[0]',
        '--eval.batch_size=1',
        f'--eval.n_episodes={n_ep}',
        '--env.max_parallel_tasks=1',
        f'--output_dir={eval_out}',
    ]
    candidates: list[list[str]] = []
    if shutil.which('lerobot-eval', path=vla_bin):
        candidates.append([f'{vla_bin}/lerobot-eval', *base_args])
    candidates.append([vla_python, '-m', 'lerobot.scripts.lerobot_eval', *base_args])

    result: EvalResult | None = None
    for cmd in candidates:
        print(f'\n=== Gold n_action_steps={n_steps} ({n_ep} ep, timeout={timeout_sec}s) ===')
        print('Command:', ' '.join(cmd))
        print(f'Streaming log: {per_log}')
        result = _run_gold_eval(cmd, per_log, timeout_sec)
        block = (
            f'=== n_action_steps={n_steps} ===\n'
            f'command: {" ".join(cmd)}\n'
            f'exit_code: {result.returncode}\n'
            f'timed_out: {result.timed_out}\n'
            f'per_config_log: {per_log}\n\n'
            f'--- combined log tail ---\n{result.stdout[-8000:]}\n'
        )
        logs.append(block)
        _write_gold_artifacts()
        _print_log_tail(per_log)
        if not result.timed_out and result.returncode == 0:
            break
        if result.timed_out:
            print(f'WARNING: gold n={n_steps} hit {timeout_sec}s timeout; continuing sweep.')
            break
        print('Failed snippet:', result.stdout[-600:])

    combined = result.stdout if result else ''
    successes, episodes, rate, metric_src = _parse_gold_success(combined, n_ep)
    if successes is None:
        s2, e2, r2, src2 = _parse_gold_from_output_dir(eval_out, n_ep)
        if s2 is not None:
            successes, episodes, rate, metric_src = s2, e2, r2, src2

    entry = {
        'n_action_steps': n_steps,
        'exit_code': None if result is None else result.returncode,
        'timed_out': bool(result and result.timed_out),
        'successes': successes,
        'episodes': episodes or n_ep,
        'success_rate': rate,
        'metric_source': metric_src,
        'output_dir': eval_out,
        'per_config_log': per_log,
    }
    gold_matrix[str(n_steps)] = entry
    sr = f'{rate:.3f}' if rate is not None else 'n/a'
    print(
        f'Gold n={n_steps}: exit={entry["exit_code"]} timed_out={entry["timed_out"]} '
        f'success={successes}/{entry["episodes"]} rate={sr} ({metric_src})'
    )
    _write_gold_artifacts()

print(f'\nGold sweep log: {gold_log}')
print(f'Gold sweep json: {gold_json}')


Cleanup: no stale VLA processes found
Gold sweep: steps=[10, 5] n_episodes=5 (n=1 capped at 3 when enabled)
Deps: /content/Longor/.vla-venv/bin/python -m pip install -q lerobot[libero]>=0.5

=== Gold n_action_steps=10 (5 ep, timeout=1800s) ===
Command: /content/Longor/.vla-venv/bin/lerobot-eval --policy.path=HuggingFaceVLA/smolvla_libero --policy.n_action_steps=10 --policy.device=cuda --env.type=libero --env.task=libero_10 --env.task_ids=[0] --eval.batch_size=1 --eval.n_episodes=5 --env.max_parallel_tasks=1 --output_dir=/content/Longor/traces/gold_lerobot_eval_n10_20260718_155117
Streaming log: /content/Longor/traces/gold_lerobot_eval_n10_20260718_155117.log
--- tail /content/Longor/traces/gold_lerobot_eval_n10_20260718_155117.log (12 lines) ---
Overall Aggregated Metrics:
{'avg_sum_reward': 0.0, 'avg_max_reward': 0.0, 'pc_success': 0.0, 'n_episodes': 5, 'eval_s': 1389.4649963378906, 'eval_ep_s': 277.8929994106293, 'video_paths': ['/content/Longor/traces/gold_lerobot_eval_n10_20260718_

In [8]:
import json
import os
import re
import subprocess
import sys
import time
import urllib.request
from datetime import datetime

N_ACTION_STEPS_SWEEP = [int(x) for x in os.environ.get('M7_SWEEP_STEPS', '10,5').split(',') if x.strip()]
N_RUNS = int(os.environ.get('M7_N_LONGOR_RUNS', os.environ.get('M7_N_EPISODES', '5')))
print(f'Longor sweep: steps={N_ACTION_STEPS_SWEEP} n_runs={N_RUNS}')

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
longor_log = f'{trace_dir}/longor_grasp_smoke_sweep_{stamp}.txt'
longor_json = f'{trace_dir}/longor_grasp_smoke_sweep_{stamp}.json'
os.environ['LONGOR_GRASP_LOG'] = longor_log
os.environ['LONGOR_SWEEP_JSON'] = longor_json

vla_python = '/content/Longor/.vla-venv/bin/python'
VLA_LOG_SWEEP = '/tmp/m7_vla_smolvla_sweep.log'
POLL_SEC = 15
MAX_WAIT_SEC = int(os.environ.get('VLA_STARTUP_MAX_WAIT_SEC', 120))

PARITY_ENV = {
    'VLA_USE_ENV_TASK_SUCCESS': 'false',
    'VLA_GRIPPER_MODE': 'absolute_open_close',
    'VLA_ACTION_SCALE': '1.0',
    'LIBERO_EE_DELTA_SCALE': '1.0',
    'LIBERO_VLA_IMAGE_FLIP': '1',
    'LIBERO_OBSERVATION_HEIGHT': '256',
    'LIBERO_OBSERVATION_WIDTH': '256',
    'VLA_STEPS_PER_ACTION': '1',
    'VLA_ADAPTIVE_SCALE': 'false',
    'VLA_STAGNATION_CHUNKS': '20',
    'LIBERO_SETTLE_STEPS': '10',
    'VLA_MAX_INNER_STEPS': '200',
    'VLA_MAX_TOOL_SECONDS': '180',
    'VLA_SIM_PICKCUBE_GRIPPER_PROFILE': 'false',
    'VLA_FAIL_WORSENING_STEPS': '200',
    'VLA_PROXIMITY_GRASP': 'false',
    'VLA_GRIPPER_CONFIDENCE': '0.5',
    'VLA_FAIL_DIST_CONSECUTIVE_STEPS': '200',
    'VLA_FAIL_IK_SATURATION_STEPS': '200',
    'LIBERO_VLA_STATE_MODE': 'eef_axis_angle',
    'LIBERO_VLA_STATE_DEBUG': '0',
    'VLA_DEBUG_ACTION_TELEMETRY': '0',
    'PYTHONPATH': '/content/Longor/src',
    'MUJOCO_GL': 'egl',
    'PYOPENGL_PLATFORM': 'egl',
}

SMOKE_RESULT_RE = re.compile(
    r'Smoke result:\s*reason=(\S+)\s+steps=(\S+)\s+min_dist=(\S+)\s+grasp_confirmed=(\S+)',
    re.I,
)
PASS_RE = re.compile(r'\bPASS:', re.I)


def _stop_vla_local(proc, log_handle):
    if log_handle and not log_handle.closed:
        log_handle.flush()
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)
    if log_handle and not log_handle.closed:
        log_handle.close()


def _start_vla_server(chunk_steps: int):
    try:
        _cleanup_stale_vla_processes()
    except NameError:
        subprocess.run(['fuser', '-k', '8765/tcp'], capture_output=True)
    log_handle = open(VLA_LOG_SWEEP, 'w', buffering=1)
    env_for_server = {
        **os.environ,
        'PYTHONUNBUFFERED': '1',
        'VLA_POLICY': 'smolvla',
        'SMOLVLA_CHECKPOINT': 'HuggingFaceVLA/smolvla_libero',
        'VLA_CHUNK_STEPS': str(chunk_steps),
    }
    proc = subprocess.Popen(
        [vla_python, 'services/run_vla_server.py'],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        cwd='/content/Longor',
        env=env_for_server,
    )
    start = time.monotonic()
    while time.monotonic() - start < MAX_WAIT_SEC:
        time.sleep(POLL_SEC)
        if proc.poll() is not None:
            _stop_vla_local(proc, log_handle)
            raise RuntimeError(f'VLA server exited early (code {proc.returncode})')
        try:
            resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
            body = json.loads(resp.read())
            if body.get('model') == 'smolvla':
                print(f'VLA server ready (VLA_CHUNK_STEPS={chunk_steps})')
                return proc, log_handle
        except Exception:
            print(f'Waiting for VLA /health ({time.monotonic() - start:.0f}s)...')
    _stop_vla_local(proc, log_handle)
    raise RuntimeError(f'VLA server not healthy within {MAX_WAIT_SEC}s')


def _parse_smoke_run(text: str, exit_code: int) -> dict:
    run = {
        'exit_code': exit_code,
        'passed': exit_code == 0,
        'reason': None,
        'min_dist': None,
        'grasp_confirmed': None,
    }
    for m in SMOKE_RESULT_RE.finditer(text):
        run['reason'] = m.group(1)
        run['min_dist'] = None if m.group(3) == 'None' else float(m.group(3))
        gc = m.group(4)
        run['grasp_confirmed'] = gc.lower() in ('true', '1') if gc != 'None' else None
    if PASS_RE.search(text):
        run['passed'] = True
    return run


logs: list[str] = []
longor_matrix: dict[str, dict] = {}
pytest_cmd_base = [
    sys.executable, '-m', 'pytest', '-m', 'integration',
    'tests/test_libero_integration.py::test_grasp_smoke', '-v', '-s',
]

for n_steps in N_ACTION_STEPS_SWEEP:
    vla_proc_sweep, vla_log_sweep = _start_vla_server(n_steps)
    runs: list[dict] = []
    try:
        for run_idx in range(N_RUNS):
            run_env = {
                **os.environ,
                **PARITY_ENV,
                'VLA_CHUNK_STEPS': str(n_steps),
                'ROBOCREW_SIM_SEED': str(run_idx),
            }
            result = subprocess.run(
                pytest_cmd_base,
                cwd='/content/Longor',
                env=run_env,
                capture_output=True,
                text=True,
                timeout=900,
            )
            combined = (result.stdout or '') + '\n' + (result.stderr or '')
            parsed = _parse_smoke_run(combined, result.returncode)
            parsed['run_idx'] = run_idx
            runs.append(parsed)
            block = (
                f'=== longor n_action_steps={n_steps} run={run_idx} ===\n'
                f'exit_code: {result.returncode}\n'
                f'parsed: {json.dumps(parsed)}\n\n'
                f'--- stdout ---\n{result.stdout}\n--- stderr ---\n{result.stderr}\n'
            )
            logs.append(block)
            with open(longor_log, 'w', encoding='utf-8') as f:
                f.write('\n\n'.join(logs))
            print(
                f'Longor n={n_steps} run={run_idx}: '
                f'{"PASS" if parsed["passed"] else "FAIL"} '
                f'reason={parsed["reason"]} min_dist={parsed["min_dist"]} '
                f'grasp_confirmed={parsed["grasp_confirmed"]}'
            )
    finally:
        _stop_vla_local(vla_proc_sweep, vla_log_sweep)

    passes = sum(1 for r in runs if r['passed'])
    longor_matrix[str(n_steps)] = {
        'n_action_steps': n_steps,
        'runs': N_RUNS,
        'passes': passes,
        'pass_rate': passes / N_RUNS if N_RUNS else 0.0,
        'episodes': runs,
    }
    print(f'Longor n={n_steps}: {passes}/{N_RUNS} pass rate={passes / N_RUNS:.3f}')

with open(longor_json, 'w', encoding='utf-8') as f:
    json.dump({'stamp': stamp, 'n_runs': N_RUNS, 'configs': longor_matrix}, f, indent=2)
print(f'\nLongor sweep log: {longor_log}')
print(f'Longor sweep json: {longor_json}')


Longor sweep: steps=[10, 5] n_runs=5
Cleanup: no stale VLA processes found
Waiting for VLA /health (15s)...
Waiting for VLA /health (30s)...
VLA server ready (VLA_CHUNK_STEPS=10)
Longor n=10 run=0: FAIL reason=max_inner_steps min_dist=0.0981 grasp_confirmed=None
Longor n=10 run=1: FAIL reason=max_inner_steps min_dist=0.1029 grasp_confirmed=None
Longor n=10 run=2: FAIL reason=max_inner_steps min_dist=0.0567 grasp_confirmed=None
Longor n=10 run=3: FAIL reason=max_inner_steps min_dist=0.0618 grasp_confirmed=None
Longor n=10 run=4: FAIL reason=max_inner_steps min_dist=0.0556 grasp_confirmed=None
Longor n=10: 0/5 pass rate=0.000
Cleanup: no stale VLA processes found
Waiting for VLA /health (15s)...
Waiting for VLA /health (30s)...
VLA server ready (VLA_CHUNK_STEPS=5)
Longor n=5 run=0: FAIL reason=max_inner_steps min_dist=0.0823 grasp_confirmed=None
Longor n=5 run=1: FAIL reason=stagnation min_dist=0.0729 grasp_confirmed=None
Longor n=5 run=2: FAIL reason=max_inner_steps min_dist=0.0452 gras

In [9]:
import glob
import json
import os

trace_dir = '/content/Longor/traces'
N_ACTION_STEPS_SWEEP = [int(x) for x in os.environ.get('M7_SWEEP_STEPS', '10,5').split(',') if x.strip()]
GOLD_NONZERO_EPS = 0.0


def _load_json(env_key: str, pattern: str) -> dict | None:
    explicit = os.environ.get(env_key, '').strip()
    if explicit and os.path.exists(explicit):
        with open(explicit, encoding='utf-8') as f:
            return json.load(f)
    matches = sorted(glob.glob(f'{trace_dir}/{pattern}'))
    if not matches:
        return None
    with open(matches[-1], encoding='utf-8') as f:
        return json.load(f)


gold_data = _load_json('GOLD_EVAL_JSON', 'gold_lerobot_eval_sweep_*.json')
longor_data = _load_json('LONGOR_SWEEP_JSON', 'longor_grasp_smoke_sweep_*.json')

if not gold_data or not longor_data:
    raise RuntimeError('Missing sweep JSON — run gold and Longor sweep cells first.')

gold_cfg = gold_data.get('configs', {})
longor_cfg = longor_data.get('configs', {})

print('=== Sweep matrix (LIBERO-10 task 0) ===')
print(f"{'n_action_steps':>14} | {'gold_success':>14} | {'longor_pass':>14} | notes")
print('-' * 65)

any_gold_nonzero = False
all_gold_near_zero = True
longor_all_zero = True

for n in N_ACTION_STEPS_SWEEP:
    key = str(n)
    g = gold_cfg.get(key, {})
    l = longor_cfg.get(key, {})
    g_succ = g.get('successes')
    g_ep = g.get('episodes')
    g_rate = g.get('success_rate')
    l_pass = l.get('passes', 0)
    l_runs = l.get('runs', 0)
    l_rate = l.get('pass_rate', 0.0)

    g_str = 'n/a'
    if g_succ is not None and g_ep:
        g_str = f'{g_succ}/{g_ep} ({g_rate:.3f})'
    l_str = f'{l_pass}/{l_runs} ({l_rate:.3f})'

    note = []
    if g.get('timed_out'):
        note.append('gold_timeout')
        all_gold_near_zero = False
    elif g.get('exit_code') not in (0, None):
        note.append(f"gold_exit={g.get('exit_code')}")
    if g_rate is not None and g_rate > GOLD_NONZERO_EPS:
        any_gold_nonzero = True
        all_gold_near_zero = False
    elif g_rate is None:
        all_gold_near_zero = False
        note.append('gold_metric_missing')
    if l_pass > 0:
        longor_all_zero = False

    print(f'{n:>14} | {g_str:>14} | {l_str:>14} | {"; ".join(note) or "-"}')

print('\n=== Per-run Longor telemetry (last run per config) ===')
for n in N_ACTION_STEPS_SWEEP:
    eps = longor_cfg.get(str(n), {}).get('episodes') or []
    if not eps:
        continue
    last = eps[-1]
    print(
        f"n={n} last: passed={last.get('passed')} reason={last.get('reason')} "
        f"min_dist={last.get('min_dist')} grasp_confirmed={last.get('grasp_confirmed')}"
    )

print('\n=== VERDICT ===')
gold_exits_ok = all(
    (
        gold_cfg.get(str(n), {}).get('exit_code') == 0
        and not gold_cfg.get(str(n), {}).get('timed_out')
    )
    for n in N_ACTION_STEPS_SWEEP
    if str(n) in gold_cfg
)
if any_gold_nonzero and longor_all_zero:
    VERDICT = 'INTEGRATION_DRIFT'
    detail = (
        'At least one gold n_action_steps config shows non-zero success while Longor '
        'passes 0/N on all configs → Longor obs/action/queue wiring likely wrong.'
    )
elif all_gold_near_zero and gold_exits_ok:
    VERDICT = 'CHECKPOINT_RECIPE'
    detail = (
        'All gold configs report ~0% success with exit 0 → SmolVLA checkpoint/recipe '
        'issue; do not keep patching Longor integration blindly.'
    )
else:
    VERDICT = 'INCONCLUSIVE'
    detail = (
        'Mixed or incomplete gold metrics (timeouts, missing metrics, or partial sweep) '
        '— inspect per-config logs under traces/gold_lerobot_eval_n*_<stamp>.log before concluding.'
    )

print(f'VERDICT: {VERDICT}')
print(detail)
os.environ['M7_SWEEP_VERDICT'] = VERDICT

if VERDICT == 'INTEGRATION_DRIFT':
    print('\nNext: run §6d integration-drift confirm slice.')
elif VERDICT == 'CHECKPOINT_RECIPE':
    print('\nStop integration churn; revisit SmolVLA training/eval recipe or checkpoint.')


=== Sweep matrix (LIBERO-10 task 0) ===
n_action_steps |   gold_success |    longor_pass | notes
-----------------------------------------------------------------
            10 |    0/5 (0.000) |    0/5 (0.000) | -
             5 |    0/5 (0.000) |    0/5 (0.000) | -

=== Per-run Longor telemetry (last run per config) ===
n=10 last: passed=False reason=max_inner_steps min_dist=0.0556 grasp_confirmed=None
n=5 last: passed=False reason=stagnation min_dist=0.0773 grasp_confirmed=None

=== VERDICT ===
VERDICT: CHECKPOINT_RECIPE
All gold configs report ~0% success with exit 0 → SmolVLA checkpoint/recipe issue; do not keep patching Longor integration blindly.

Stop integration churn; revisit SmolVLA training/eval recipe or checkpoint.


In [ ]:
import json
import os
import subprocess
import sys
import time
import urllib.request

VERDICT = os.environ.get('M7_SWEEP_VERDICT', '').strip()
if VERDICT != 'INTEGRATION_DRIFT':
    print(f'Skip §6d — M7_SWEEP_VERDICT={VERDICT or "(unset)"} (not INTEGRATION_DRIFT).')
else:
    subprocess.run(['git', 'pull', '--ff-only'], check=True, cwd='/content/Longor')

    gold_json = os.environ.get('GOLD_EVAL_JSON', '')
    best_n = 10
    if gold_json and os.path.exists(gold_json):
        with open(gold_json, encoding='utf-8') as f:
            gold_data = json.load(f)
        best_n = int(max(
            gold_data.get('configs', {}).items(),
            key=lambda kv: (kv[1].get('success_rate') or 0.0, -int(kv[0])),
        )[0])

    N_RUNS = 5
    print(f'§6d confirm slice: n_action_steps={best_n}, {N_RUNS} runs after integration fix.')

    # Inline minimal Longor rerun (same PARITY_ENV as §6b).
    import re
    from datetime import datetime

    trace_dir = '/content/Longor/traces'
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    confirm_log = f'{trace_dir}/longor_confirm_{stamp}.txt'
    vla_python = '/content/Longor/.vla-venv/bin/python'
    PARITY_ENV = {
        'VLA_USE_ENV_TASK_SUCCESS': 'false',
        'VLA_GRIPPER_MODE': 'absolute_open_close',
        'VLA_ACTION_SCALE': '1.0',
        'LIBERO_EE_DELTA_SCALE': '1.0',
        'LIBERO_VLA_IMAGE_FLIP': '1',
        'LIBERO_OBSERVATION_HEIGHT': '256',
        'LIBERO_OBSERVATION_WIDTH': '256',
        'VLA_STEPS_PER_ACTION': '1',
        'VLA_ADAPTIVE_SCALE': 'false',
        'VLA_STAGNATION_CHUNKS': '20',
        'VLA_MAX_INNER_STEPS': '200',
        'VLA_MAX_TOOL_SECONDS': '180',
        'VLA_SIM_PICKCUBE_GRIPPER_PROFILE': 'false',
        'VLA_PROXIMITY_GRASP': 'false',
        'VLA_GRIPPER_CONFIDENCE': '0.5',
        'VLA_CHUNK_STEPS': str(best_n),
        'LIBERO_VLA_STATE_MODE': 'eef_axis_angle',
        'PYTHONPATH': '/content/Longor/src',
        'MUJOCO_GL': 'egl',
        'PYOPENGL_PLATFORM': 'egl',
    }
    SMOKE_RESULT_RE = re.compile(
        r'Smoke result:\s*reason=(\S+)\s+steps=(\S+)\s+min_dist=(\S+)\s+grasp_confirmed=(\S+)',
        re.I,
    )

    before = os.environ.get('LONGOR_SWEEP_JSON', '')
    before_passes = 0
    if before and os.path.exists(before):
        with open(before, encoding='utf-8') as f:
            prev = json.load(f)
        before_passes = prev.get('configs', {}).get(str(best_n), {}).get('passes', 0)

    proc = subprocess.Popen(
        [vla_python, 'services/run_vla_server.py'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        cwd='/content/Longor',
        env={**os.environ, 'VLA_POLICY': 'smolvla', 'VLA_CHUNK_STEPS': str(best_n)},
    )
    for _ in range(24):
        time.sleep(5)
        try:
            urllib.request.urlopen('http://localhost:8765/health', timeout=3)
            break
        except Exception:
            pass
    passes = 0
    logs = []
    for run_idx in range(N_RUNS):
        env = {**os.environ, **PARITY_ENV, 'ROBOCREW_SIM_SEED': str(run_idx)}
        r = subprocess.run(
            [sys.executable, '-m', 'pytest', '-m', 'integration',
             'tests/test_libero_integration.py::test_grasp_smoke', '-v', '-s'],
            cwd='/content/Longor', env=env, capture_output=True, text=True, timeout=900,
        )
        ok = r.returncode == 0
        passes += int(ok)
        logs.append(f'run={run_idx} exit={r.returncode}\n{r.stdout}\n{r.stderr}')
    proc.terminate()
    with open(confirm_log, 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(logs))
    print(f'Confirm delta: before={before_passes}/20 after={passes}/{N_RUNS} (n={best_n})')
    print(f'Confirm log: {confirm_log}')


## 7 — Integration smokes

In [ ]:
# With SmolVLA server already up from section 5:
!PYTHONPATH=src MUJOCO_GL=egl PYOPENGL_PLATFORM=egl pytest -m integration tests/test_libero_integration.py -v

## 8 — Grasp debug (optional)

In [ ]:
# Optional: grasp smoke with action telemetry (server must be up from §5)
import os
# SmolVLA LIBERO trains on eef_axis_angle layout; set agent_pos to A/B test if obs has it
os.environ.setdefault('LIBERO_VLA_STATE_MODE', 'eef_axis_angle')  # eef_axis_angle | agent_pos | joints_gripper
os.environ['LIBERO_VLA_STATE_DEBUG'] = '1'  # step-0 log: eef_axis_angle vs obs agent_pos
os.environ['VLA_DEBUG_ACTION_TELEMETRY'] = '1'
!PYTHONPATH=src MUJOCO_GL=egl PYOPENGL_PLATFORM=egl pytest -m integration tests/test_libero_integration.py::test_grasp_smoke -v -s


## 9 — M7 perception wiring smoke (unit tests, no sim-GT guard)

In [ ]:
import os
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m7_perception_smoke_{stamp}.txt'
print(f'Capture: {capture_path}')
os.environ['M7_CAPTURE'] = capture_path
!PYTHONPATH=src pytest tests/test_libero_real_perception.py tests/test_libero_backend.py::test_libero_backend_register_skills_real_mode tests/test_libero_backend.py::test_libero_backend_register_skills tests/test_libero_backend.py::test_libero_backend_list_objects_routes_real_perception tests/test_libero_backend.py::test_libero_backend_list_objects_real_mode_no_detection_returns_empty -m 'not integration' -v 2>&1 | tee "$M7_CAPTURE"


## 10 — Run LLM example (real perception)

In [ ]:
import os, subprocess, sys
from datetime import datetime

# Set M7_STRICT_ASSERTS=1 for acceptance runs; default OFF for diagnosis loops.
strict_mode = os.environ.get('M7_STRICT_ASSERTS', '').lower() in ('1', 'true', 'yes')
print(f'Strict asserts: {"ON" if strict_mode else "OFF"} (set M7_STRICT_ASSERTS=1 for acceptance)')

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m7_llm_{stamp}.txt'
run_log_path = f'{trace_dir}/m7_llm_{stamp}.log'

result = subprocess.run(
    [sys.executable, 'examples/long_horizon_llm_libero.py'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=600,
    env={
        **os.environ,
        'PYTHONPATH': '/content/Longor/src',
        'MUJOCO_GL': 'egl',
        'PYOPENGL_PLATFORM': 'egl',
        'PERCEPTION_BACKEND': 'real',
        'VLA_POLICY': 'smolvla',
        'LONGOR_RUN_LOG': run_log_path,
        'LOG_LEVEL': 'DEBUG',
        'HF_HUB_DISABLE_PROGRESS_BARS': '1',
        'TQDM_DISABLE': '1',
        # Section 7 fast-fail profile: keep retries useful, avoid 10+ minute dead loops.
        'LONGOR_MAX_REPLANS': '2',
        'VLA_MAX_TOOL_SECONDS': '45',
        'VLA_MAX_INNER_STEPS': '80',
        'VLA_STAGNATION_CHUNKS': '8',
        'VLA_FAIL_WORSENING_STEPS': '40',
        'VLA_FAIL_DIST_CONSECUTIVE_STEPS': '40',
        'VLA_FAIL_IK_SATURATION_STEPS': '20',
    },
)
combined = result.stdout + result.stderr
with open(capture_path, 'w') as f:
    f.write(combined)
print(f'Capture: {capture_path}')
print(f'Run log: {run_log_path}')
print('=== Key lines ===')
for line in combined.splitlines():
    print(line)
print('Exit code:', result.returncode)
_log_source = combined
if os.path.exists(run_log_path):
    with open(run_log_path) as _f:
        _log_source = combined + '\n' + _f.read()

_checks = [
    (result.returncode == 0, f'exit code {result.returncode}'),
    ('ImportError' not in combined, 'Import failure detected'),
    ('ModuleNotFoundError' not in combined, 'Module not found'),
    ('Task succeeded:' in _log_source, 'E2E finished but task was not marked succeeded'),
    (
        'Run summary:' in _log_source and 'success=True' in _log_source,
        'Expected Run summary with success=True',
    ),
]
_failures = [msg for ok, msg in _checks if not ok]
if _failures:
    print('=== Acceptance checks FAILED ===')
    for msg in _failures:
        print(f'  - {msg}')
    for line in _log_source.splitlines():
        if 'Run summary:' in line or 'Task succeeded:' in line:
            print(f'  {line}')
    if strict_mode:
        raise AssertionError('; '.join(_failures))
    print('Continuing to diagnostics (strict OFF). Set M7_STRICT_ASSERTS=1 for acceptance.')
else:
    print('M7 LLM example PASS')

## 11 — Trace summary

In [ ]:
import glob, json, os

trace_dir = '/content/Longor/traces'
files = sorted(glob.glob(f'{trace_dir}/*.jsonl'))
if not files:
    print('No trace files found')
else:
    print(f'Latest trace: {files[-1]}')
    with open(files[-1]) as f:
        events = [json.loads(l) for l in f if l.strip()]
    print(f'Total events: {len(events)}')
    for e in events:
        evt = e.get('event')
        skill = e.get('skill', '')
        step = e.get('step', '')
        if evt == 'plan_start':
            skills = e.get('skills')
            if skills:
                print(f"Plan skills: {' → '.join(skills)}")
        if skill in ('arm.grab', 'arm.place'):
            r = {}
            try:
                r = json.loads(e.get('result', '{}'))
            except Exception:
                pass
            print(f"[step {step}] {skill}: status={r.get('status')} reason={r.get('reason')} steps_applied={r.get('steps_applied')} min_dist={r.get('min_dist')} last_raw={r.get('last_raw_action')}")
        elif evt == 'skill_call' and skill not in ('arm.grab', 'arm.place'):
            print(f"[step {step}] {skill}: {str(e.get('result',''))[:120]}")
        elif evt == 'feedback':
            reason = e.get('reason')
            reason_part = f' — {reason}' if reason else ''
            print(f"[step {step}] feedback {e.get('description','')!r} → {e.get('decision')}{reason_part}")
        elif evt in ('replan', 'plan_end'):
            print(json.dumps(e, indent=2))

run_logs = sorted(glob.glob(f'{trace_dir}/m7_llm_*.log'))
if run_logs:
    latest_log = run_logs[-1]
    print(f'Latest section-10 run log: {latest_log}')
    with open(latest_log) as f:
        print(f.read()[-4000:])
else:
    print('No m7_llm_*.log files found under traces')

## 12 — Pytest (non-integration)

In [ ]:
import os
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m7_pytest_{stamp}.txt'
print(f'Capture: {capture_path}')
os.environ['M7_CAPTURE'] = capture_path
!PYTHONPATH=src pytest -m 'not integration' --tb=short -q 2>&1 | tee "$M7_CAPTURE"


## 13 — Teardown

In [ ]:
vla_proc.terminate()
print('VLA server stopped.')